## 0. Evvelce ümumi fikir: maşın öyrenmesi ve reqressiya nedir

Maşın öyrenmesi kompütere qaydaları el ile yazmaq evezine, ona çoxlu nümune gösterib qaydaları özünün tapmasını temin etmekdir. Meselen, biz kompütere "sahesi 100 kv.m olan ev 90 min manat edir" deye yazmırıq. Onun evezine minlerle satılmış evin melumatını veririk, kompüter de sahe, otaq sayı, yaş kimi göstericilerle qiymet arasındakı elaqeni özü keşf edir.

Reqressiya bu işin o növüdür ki, proqnozlaşdırdığımız şey bir reqemdir: qiymet, temperatur, maaş, satış miqdarı. Cavab kateqoriya olarsa (meselen, "spam/spam deyil"), bu artıq tesnifat adlanır. Bu tapşırıqdakı bütün alqoritmler hem reqressiya, hem de tesnifat üçün işleyir, biz ise reqressiya variantına baxırıq.

Her model işleyerken iki merheleden keçir. Birinci merhelede (telim) model köhne melumatlara baxıb öyrenir. İkinci merhelede (test) onun heç vaxt görmediyi yeni evlerin qiymetini nece tapdığına baxırıq. Test hissesini ayrıca saxlamağımızın sebebi budur: imtahanda dersde hell olunmuş suallar düşseydi, şagirdin heqiqeten öyrenib-öyrenmediyini bile bilmezdik.

In [ ]:

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


np.random.seed(42)
print("Kitabxanalar hazırdır")

### Nümune melumatı yaradırıq

İnternetden melumat yüklemekle vaxt itirmeyek deye, ev qiymetleri melumatını özümüz süni şekilde yaradırıq. Qiymet bele formalaşır: sahe artdıqca qiymet artır, ev köhneldikce ve merkezden uzaqlaşdıqca qiymet düşür. Elave olaraq, böyük ve merkeze yaxın evlere ekstra "premium" qiymet veririk. Bu, heyatda tez-tez rastlaşdığımız "qaydalar qarışığı" veziyyetidir ve ağac esaslı modellerin bacardığı şeydir. Sonda her qiymete tesadüfi kiçik "ses-küy" elave edirik ki, real heyat kimi bir az qarışıq olsun.

In [ ]:
n = 1500  # ev sayı

sahe = np.random.uniform(40, 200, n)                                  # kv.metr
otaq = np.clip(np.round(sahe / 40 + np.random.normal(0, 0.7, n)), 1, 7)  # otaq sayı
yas = np.random.uniform(0, 50, n)                                      # evin yaşı (il)
merkeze = np.random.uniform(0, 20, n)                                  # merkeze mesafe (km)

qiymet = 800 * sahe + 12000 * otaq - 1200 * yas - 3000 * merkeze
qiymet = qiymet + np.where((sahe > 120) & (merkeze < 5), 40000, 0)    # premium bonus
qiymet = qiymet + np.random.normal(0, 15000, n)                        # ses-küy

df = pd.DataFrame({
    "sahe_m2": sahe,
    "otaq_sayi": otaq,
    "evin_yasi": yas,
    "merkeze_mesafe_km": merkeze,
    "qiymet": qiymet,
})

X = df.drop(columns="qiymet")   # melumatlar (giriş)
y = df["qiymet"]                # tapmaq istediyimiz cavab

# 80% telim, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Telim üçün ev sayı:", len(X_train), "| Test üçün ev sayı:", len(X_test))
df.head()

In [ ]:
# Her model üçün eyni qaydada netice çıxaran kiçik kömekçi funksiya
def qiymetlendir(model, ad):
    bas = time.time()
    model.fit(X_train, y_train)
    muddet = time.time() - bas

    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    return {
        "Model": ad,
        "Train R2": r2_score(y_train, pred_train),
        "Test R2": r2_score(y_test, pred_test),
        "MAE": mean_absolute_error(y_test, pred_test),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred_test)),
        "Telim vaxtı (san)": muddet,
    }

# 1. Maşın öyrenmesi alqoritmleri

## 1.1. Decision Tree (Qerar ağacı)

**Prinsipi.** Qerar ağacı "20 sual" oyununa benzeyir. Bir dostunuz kiminse fikrinde tutduğu şeyi tapmaq üçün "beli/xeyr" cavablı suallar verir ve her cavabdan sonra imkanlar daralır. Ağac da melumatlara baxıb en yaxşı sualı özü seçir: "Sahe 100-den böyükdürmü?" Cavab beli olarsa, bir budağa gedir, xeyr olarsa, digerine. Sonra her budaqda yeni sual verir: "Merkeze mesafe 5 km-den azdırmı?" Belece suallar zenciri qurulur ve sonda "yarpaq" adlanan sonuncu qutulara çatırıq. Her yarpaqda hemin qutuya düşen evlerin orta qiymeti proqnoz kimi yazılır.

Ağac sualları nece seçir? Her addımda bütün mümkün sualları sınayır ve qiymetleri bir-birine en çox yaxın olan qruplar yaradan sualı götürür. Reqressiyada bunun ölçüsü qrup daxilindeki xetanın (dispersiyanın) en çox azalmasıdır.

**Esas üstünlükleri.** En böyük üstünlüyü başa düşülen olmasıdır: ağacı çekib bankdakı ve ya hekimdeki hemkarınıza göstere bilersiniz, o da "model niye bele dedi" sualının cavabını görer. Melumatı xüsusi hazırlamaq (miqyaslaşdırmaq, normallaşdırmaq) lazım olmur. Hem reqemli, hem kateqoriyalı melumatla işleye bilir. Xetti olmayan elaqeleri (meselen, "sahe böyükdürse ve merkeze yaxındırsa bonus var") asanlıqla tutur. Telimi de süretlidir.

**Esas çatışmazlıqları.** En böyük problemi hedden artıq öyrenmekdir (overfitting). Ağacı derinleşmeye qoysanız, her evi tek-tek ezberleyen yarpaqlar yaradır. Telim melumatında ela görünür, yeni melumatda ise pis netice verir. Bu, metni başa düşmeden ezberleyen telebeye benzeyir. Diger problemi qeyri-sabitliyidir: melumatda kiçik deyişiklik bütün ağacın formasını deyişe biler. Bir de proqnozlar pilleli olur, yeni qiymet hamar deyişmir, qutudan qutuya tullanır.

**Tetbiq nümuneleri.** Bankda kredit verilib-verilmemesi qerarını izah etmek, sade tibbi diaqnostika qaydaları, müşterinin xidmeti terk edib-etmeyeceyini texmin etmek, qiymet ve ya telebat üçün sade qaydalar çıxarmaq.

Aşağıda evvelce kiçik (derinliyi 3 olan) ağacı çekirik ki, "suallar zencirini" gözünüzle göresiniz.

In [ ]:
kicik_agac = DecisionTreeRegressor(max_depth=3, random_state=42)
kicik_agac.fit(X_train, y_train)

plt.figure(figsize=(20, 8))
plot_tree(kicik_agac, feature_names=X.columns, filled=True, rounded=True, fontsize=9)
plt.title("Derinliyi 3 olan qerar ağacı (her qutuda sual ve orta qiymet var)")
plt.show()

Qutulara baxın: yuxarıdakı qutu birinci sualdır (adeten sahe ile bağlı), `samples` hemin qutuya neçe evin düşdüyünü, `value` ise hemin evlerin orta qiymetini gösterir. En aşağıdakı qutular yarpaqlardır ve proqnozlar mehz oradan gelir.

İndi overfitting-i öz gözümüzle görek. Eyni ağacı üç ferqli derinlikde qururuq: çox dayaz, orta ve limitsiz. Limitsiz ağac telim melumatını demek olar ki, mükemmel ezberleyecek, amma testde bu uğur tekrarlanmayacaq.

In [ ]:
netice = []
for derinlik in [2, 5, None]:
    agac = DecisionTreeRegressor(max_depth=derinlik, random_state=42)
    agac.fit(X_train, y_train)
    netice.append({
        "Maksimum derinlik": "limitsiz" if derinlik is None else derinlik,
        "Train R2": r2_score(y_train, agac.predict(X_train)),
        "Test R2": r2_score(y_test, agac.predict(X_test)),
    })

pd.DataFrame(netice)

Cedvelde limitsiz ağacın Train R² göstericisi 1-e çox yaxın olacaq, Test R² ise ondan xeyli aşağı. Bu ferq overfitting-in ölçüsüdür. Dayaz ağac ise eksine, ne telimde, ne testde çox yaxşı deyil, çünki melumatı yaxşı öyrenmeye imkanı olmur (bunu underfitting adlandırırlar). En yaxşısı ortada bir yerdedir.

Bir ağacın bu zeif cehetlerini düzeltmek üçün ağıllı fikir ortaya çıxıb: bir ağacın evezine çoxlu ağac işletmek. Növbeti iki alqoritm bu fikrin iki ferqli yoludur.

## 1.2. Random Forest (Tesadüfi meşe)

**Prinsipi.** Tutaq ki, bir evin qiymetini bilmek isteyirsiniz. Bir neferden soruşsanız, sehv ede biler. Amma 200 neferden soruşub cavabların ortalamasını götürseniz, ferdi sehvler bir-birini kompensasiya edir ve netice xeyli deqiq olur. Buna "izdihamın müdrikliyi" deyirler. Random Forest tam bu işi görür: yüzlerle qerar ağacı qurur ve onların proqnozlarının ortalamasını (reqressiyada) verir.

Amma bütün ağaclar eyni olsaydı, 200 defe eyni cavabı almış olardıq. Ona göre her ağacı ferqli etmek üçün iki tesadüfilik elave olunur. Birincisi, her ağaca telim melumatının tesadüfi seçilmiş bir hissesi verilir (seçim tekrarlı olur, yeni bir ev iki defe düşe biler, bu üsul bootstrap adlanır). İkincisi, ağac her sualı seçerken bütün göstericilere yox, yalnız tesadüfi seçilmiş bir neçesine baxır. Neticede her ağac melumatı bir az ferqli bucaqdan görür, sehvleri de ferqli olur ve ortalama götürende sehvler bir-birini söndürür. Bütün ağaclar bir-birinden asılı olmadan, paralel qurulur.

**Esas üstünlükleri.** Tek ağaca nisbeten overfitting daha az olur ve netice daha sabitdir. Çox vaxt hiperparametrleri çox tenzimlemeden de yaxşı işleyir. Hansı göstericinin daha vacib olduğunu gösterir (feature importance). Melumatı miqyaslaşdırmaq lazım deyil ve kenar qiymetlere (outlier) qarşı dözümlüdür. Ağaclar bir-birinden asılı olmadığı üçün paralel qurulur.

**Esas çatışmazlıqları.** Artıq tek bir ağac kimi asan izah olunmur, çünki 200 ağacı gözle oxumaq mümkün deyil. Model daha çox yaddaş tutur ve proqnoz vermek tek ağacdan yavaşdır. Bir de ağac esaslı bütün modeller kimi, telim melumatının ehate etdiyi aralıqdan kenara çıxa bilmir: telimde en bahalı ev 300 min idise, model 800 min proqnozlaşdırmaz.

**Tetbiq nümuneleri.** Ev ve avtomobil qiymetinin texmini, kredit riskinin qiymetlendirilmesi, xestelik riskinin proqnozu, müşteri davranışı analizi, hansı göstericinin neticeye daha çox tesir etdiyini tapmaq.

In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_netice = qiymetlendir(rf, "Random Forest")
print(rf_netice)

# Hansı gösterici qiymete daha çox tesir edir?
onem = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
onem.plot(kind="barh", figsize=(7, 3), title="Random Forest: göstericilerin ehemiyyeti")
plt.show()

Qrafikde sahenin en uzun çubuğa malik olması gözlenilen idi, çünki biz qiymeti esasen sahe üzerinde qurmuşduq. Model bunu özü, bize heç ne dememişden tapdı, bu da maşın öyrenmesinin esas fikridir.

## 1.3. Gradient Boosting (Qradiyent bustinqi)

**Prinsipi.** Random Forest-de ağaclar bir-birinden asılı olmadan, paralel qurulurdu. Gradient Boosting ise tam eks yanaşmadır: ağaclar **ardıcıl** qurulur ve her yeni ağac evvelkilerin sehvlerini düzeltmeye çalışır.

Bunu qolf oyunu ile tesevvür ede bilersiniz. İlk zerbe topu deşiye yaxınlaşdırır, amma tam salmır. İkinci zerbe qalan mesafeni azaldır. Üçüncü zerbe daha da yaxınlaşdırır. Her zerbe evvelkinin qalıq xetasına baxıb ona uyğun düzeliş edir. Gradient Boosting de beledir:

Birinci addımda çox sade bir proqnoz veririk, adeten bütün evler üçün orta qiymeti. Sonra her evde sehvi hesablayırıq (heqiqi qiymet minus proqnoz), buna qalıq (residual) deyilir. İkinci addımda kiçik, sade bir ağac qururuq ve ondan bu **sehvleri** proqnozlaşdırmasını isteyirik. Sonra hemin ağacın cavabını kiçik bir çekiyle (learning rate) evvelki proqnoza elave edirik ve proqnoz bir az yaxşılaşır. Yene qalıq xetanı hesablayıb növbeti ağacı qururuq. Bu proses yüzlerle defe tekrarlanır.

"Qradiyent" sözü ona göre var ki, bu proses riyazi olaraq xeta funksiyasını azaltmaq üçün "en süretli enme istiqametinde" addım atmağa beraberdir. MSE xetası ile işleyende hemin istiqamet ele sadece qalıqlardır, ona göre mentiqi qalıqlarla başa düşmek kifayetdir.

Learning rate (öyrenme süreti) çox vacib parametrdir. Kiçik olarsa (meselen, 0.05), her ağac yalnız kiçik düzeliş edir, daha çox ağac lazım olur, amma netice daha deqiq ve overfitting riski daha az olur. Böyük olarsa, model süretle öyrenir, amma heddi aşa biler.

**Esas üstünlükleri.** Çox vaxt tabulyar (cedvel şeklinde) melumatda en yüksek deqiqliyi veren alqoritmlerden biridir. Xetti olmayan mürekkeb elaqeleri yaxşı öyrenir. Müxtelif xeta funksiyaları ile işleye bilir.

**Esas çatışmazlıqları.** Ağaclar ardıcıl qurulduğu üçün paralelleşdirmek çetindir ve telim Random Forest-den yavaş ola bilir. Hiperparametrlere (ağac sayı, öyrenme süreti, derinlik) hessasdır, yaxşı tenzimlemek vaxt aparır. Ağacların sayını hedden çox artırsanız, overfitting başlaya biler. Kenar qiymetlere qarşı hessasdır, çünki böyük sehvlerin düzeldilmesine çox diqqet verir.

**Tetbiq nümuneleri.** Satış ve telebatın proqnozu, sığorta ve kredit riski, axtarış neticelerinin sıralanması, qiymet proqnozu.

Aşağıda prosesi **el ile** quraq ki, "her ağac sehvi düzeldir" fikri aydın görünsün. Çox kiçik bir nümunede (8 ev, yalnız sahe) üç addım edeceyik.

In [ ]:

sahe_k = np.array([[50], [60], [80], [100], [120], [140], [160], [180]])
qiymet_k = np.array([60, 70, 85, 110, 135, 150, 185, 200])

lr = 0.5
proqnoz = np.full(len(qiymet_k), qiymet_k.mean())

print("Başlanğıc proqnoz (hamıya orta):", proqnoz[0])
print("Başlanğıc orta mütleq xeta:", round(np.mean(np.abs(qiymet_k - proqnoz)), 2))
print()

for addim in range(1, 4):
    qaliq = qiymet_k - proqnoz
    kicik = DecisionTreeRegressor(max_depth=1)
    kicik.fit(sahe_k, qaliq)
    proqnoz = proqnoz + lr * kicik.predict(sahe_k)

    print(f"{addim}-ci ağacdan sonra orta mütleq xeta:", round(np.mean(np.abs(qiymet_k - proqnoz)), 2))

Her addımdan sonra xetanın azaldığını görürsünüz. Real Gradient Boosting mehz bunu edir, sadece yüzlerle addımla ve bütün göstericilerle. İndi eyni şeyi scikit-learn-in hazır alqoritmi ile ev qiymetleri melumatında işledek.

In [ ]:
gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)
gb_netice = qiymetlendir(gb, "Gradient Boosting")
print(gb_netice)

## 1.4. XGBoost (Extreme Gradient Boosting, ekstremal qradiyent bustinqi)

**Prinsipi.** XGBoost yeni bir fikir deyil. O, eyni Gradient Boosting ideyasıdır: ağaclar ardıcıl qurulur ve her biri evvelkilerin sehvini düzeldir. Ferq ondadır ki, bu fikir çox diqqetle mühendislik baxımından tekmilleşdirilib: daha süretli, daha ağıllı ve overfitting-e qarşı daha qorunaqlı edilib. "Extreme" sözü mehz bu sebebden var. Bir neçe il Kaggle müsabiqelerinde cedvel melumatı üzre qalibiyyetlerin çoxu XGBoost-la elde edilirdi.

**Üstünlükleri.** Yüksek deqiqlik verir, çox süretlidir, hem telim, hem de proqnozda böyük melumat bazalarında yaxşı işleyir, boş (NaN) deyerleri özü idare edir, overfitting-e qarşı daxili qorumaları var. Hiperparametrler çoxdur, bu da yaxşı tenzimleme imkanı verir.

**Çatışmazlıqları.** Hiperparametrlerin çox olması yeni başlayan üçün qarışıqlıq yarada biler. Tek ağac ve hetta Random Forest qeder izah olunan deyil. Çox kiçik melumatda sade modellerden üstün olmaya biler. Şekil, ses ve metn kimi strukturlaşmamış melumatda neyron şebekeleri daha uyğundur.

**Tetbiq nümuneleri.** Fırıldaqçılığın aşkarlanması, reklama klik ehtimalının proqnozu, kredit skorinqi, satış proqnozu, Kaggle tipli cedvel meseleleri.

### Gradient Boosting ile XGBoost arasındakı ferqler (xüsusi vacib sual)

Birinci ferq **requlyarizasiyadır**. Klassik Gradient Boosting yalnız sehvi azaltmağa çalışır. XGBoost ise hedef funksiyasına cerime elave edir: ağacın çox mürekkeb olması ve yarpaq çekilerinin çox böyük olması cerimelenir (L1 ve L2 requlyarizasiya, hemçinin `gamma` parametri ile lazımsız bölünmelerin kesilmesi). Bu, modeli sade saxlayır ve overfitting-in qarşısını alır. Bu, XGBoost-un en mühüm ferqidir.

İkinci ferq **riyazi deqiqlikdir**. Gradient Boosting yalnız birinci töremeden (qradiyentden) istifade edir, yeni "hansı istiqamete getmek lazımdır" sualına cavab verir. XGBoost ikinci töremeden (hessian) de istifade edir, yeni eyrinin nece eyildiyini de nezere alır. Bunu dağdan enmekle müqayise ede bilersiniz: biri yalnız yamacın meyline baxır, digeri hem meyli, hem de yamacın ne qeder keskin deyişdiyini görür, ona göre daha ağıllı addım ata bilir.

Üçüncü ferq **süret ve mühendislikdir**. XGBoost ağac qurarken bölünme nöqtelerini axtarmağı paralel aparır (ağacların özü ardıcıl qalır, amma bir ağacın daxilindeki hesablamalar paylanır), yaddaşdan semereli istifade edir ve histoqram esaslı süretli üsul teklif edir. Ona göre böyük melumatda adeten xeyli süretlidir.

Dördüncü ferq **boş deyerlerle işlemekdir**. XGBoost NaN olan xanalarda hansı budağa getmeyin daha yaxşı olduğunu telim zamanı özü öyrenir. Klassik scikit-learn Gradient Boosting-de ise bu uzun müddet desteklenmirdi, ona göre evvelceden boşluqları doldurmaq lazım gelirdi.

Beşinci ferq **elave imkanlardır**. XGBoost-da daxili cross-validation, erken dayanma (early stopping: yaxşılaşma dayananda telim özü dayanır), her ağac üçün melumatın ve göstericilerin tesadüfi bir hissesini götürmek (`subsample`, `colsample_bytree`) kimi hazır aletler var. Bunlar hem süreti, hem de ümumileşdirme qabiliyyetini yaxşılaşdırır.

Qısa desek: **Gradient Boosting ideyadır, XGBoost ise hemin ideyanın cerimeleme, ikinci töreme ve süret baxımından güclendirilmiş versiyasıdır.**

In [ ]:
xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=3,
                   reg_lambda=1.0, subsample=0.9, colsample_bytree=0.9,
                   random_state=42, n_jobs=-1)
xgb_netice = qiymetlendir(xgb, "XGBoost")
print(xgb_netice)

İndi boş deyerler ferqini canlı yoxlayaq. Telim melumatında evin yaşının 10 faizini silirik ve her iki modeli telim etmeye çalışırıq.

In [ ]:
X_bosluqlu = X_train.copy()
silinecek = X_bosluqlu.sample(frac=0.10, random_state=1).index
X_bosluqlu.loc[silinecek, "evin_yasi"] = np.nan
print("Boş xana sayı:", int(X_bosluqlu.isna().sum().sum()))


XGBRegressor(n_estimators=50).fit(X_bosluqlu, y_train)
print("XGBoost: boşluqlu melumatla telim oldu")


try:
    GradientBoostingRegressor(n_estimators=50).fit(X_bosluqlu, y_train)
    print("Gradient Boosting: bu versiyada boşluqları qebul etdi")
except ValueError:
    print("Gradient Boosting: boş deyerlere göre xeta verdi (evvelceden doldurmaq lazımdır)")

## 1.5. Dörd alqoritmin bir-biri ile müqayisesi

Dörd alqoritmi bir fikirle ayırd etmek olar. **Decision Tree** tek bir ağacdır: sade, izah olunan, amma overfitting-e meyilli. **Random Forest** çoxlu ağacı paralel qurub ortalamasını alır ve ferdi sehvleri bir-birine kompensasiya etdirir. **Gradient Boosting** ağacları ardıcıl qurur ve her biri evvelkinin sehvini düzeldir. **XGBoost** ise Gradient Boosting-in cerimeleme, ikinci töreme ve süret baxımından güclendirilmiş formasıdır.

Random Forest ile Gradient Boosting arasındakı esas ferqi ise bele xatırlayın: Random Forest-de ağaclar bir-birinden bixeberdir ve **paralel** işleyir, ortalamaya baxır. Boosting-de ağaclar bir-birine **bağlıdır**, ardıcıl işleyir ve her biri evvelkinin sehvini düzeldir. Random Forest esasen "dispersiyanı" (qeyri-sabitliyi) azaldır, boosting ise "meyli" (sistematik sehvi) azaldır.

İndi dördünü eyni melumatda yarışdıraq.

In [ ]:
dt = DecisionTreeRegressor(max_depth=6, random_state=42)

dt_netice = qiymetlendir(dt, "Decision Tree")

rf_netice = qiymetlendir(RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1), "Random Forest")
gb_netice = qiymetlendir(GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42), "Gradient Boosting")
xgb_netice = qiymetlendir(XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=3,
                                       reg_lambda=1.0, subsample=0.9, colsample_bytree=0.9,
                                       random_state=42, n_jobs=-1), "XGBoost")

yekun = pd.DataFrame([dt_netice, rf_netice, gb_netice, xgb_netice]).set_index("Model")
yekun.round(3)

Cedveli oxuyanda bunlara baxın. Test R² ne qeder 1-e yaxındırsa, model bir o qeder yaxşıdır. MAE ve RMSE ne qeder kiçikdirse, bir o qeder yaxşıdır. Train R² ile Test R² arasındakı ferq böyükdürse, model ezberleyib demekdir. Adeten tek Decision Tree en zeif netice verir, ansambl modelleri (Random Forest, Gradient Boosting, XGBoost) ise xeyli yaxşı olur. Bizim melumatın kiçik ve süni olduğunu unutmayın: real melumatda sıralama deyişe biler, ona göre "hemişe XGBoost qalib gelir" demek düzgün olmaz.

Aşağıda her modelin proqnozunu heqiqi qiymetle müqayise eden qrafikler var. Nöqteler ne qeder diaqonal xettin etrafında toplanıbsa, proqnoz bir o qeder deqiqdir.

In [ ]:
modeller = {
    "Decision Tree": DecisionTreeRegressor(max_depth=6, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42, n_jobs=-1),
}

fig, oxlar = plt.subplots(1, 4, figsize=(20, 4.5), sharex=True, sharey=True)
for ox, (ad, model) in zip(oxlar, modeller.items()):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    ox.scatter(y_test, pred, s=8, alpha=0.5)
    sinir = [y_test.min(), y_test.max()]
    ox.plot(sinir, sinir, color="red", linestyle="--")   # ideal xett
    ox.set_title(f"{ad}\nR2 = {r2_score(y_test, pred):.3f}")
    ox.set_xlabel("Heqiqi qiymet")
oxlar[0].set_ylabel("Proqnoz")
plt.tight_layout()
plt.show()

# 2. Reqressiya metrikleri

Model proqnoz verende en vacib sual budur: bu proqnoz ne qeder yaxşıdır? Cavab üçün proqnozları heqiqi qiymetlerle müqayise edib bir reqeme yığırıq. Bu reqemlere metrik deyilir. Dörd esas reqressiya metrikini eyni sade nümune üzerinde öyrenek.

Nümunede 5 evin heqiqi qiymeti ve modelin proqnozu var (min manatla). Her evde **xeta = heqiqi qiymet − proqnoz**.

| Ev | Heqiqi | Proqnoz | Xeta | Xetanın mütleq deyeri | Xetanın kvadratı |
|----|--------|---------|------|-----------------------|------------------|
| 1 | 100 | 110 | -10 | 10 | 100 |
| 2 | 200 | 190 | 10 | 10 | 100 |
| 3 | 300 | 330 | -30 | 30 | 900 |
| 4 | 400 | 380 | 20 | 20 | 400 |
| 5 | 500 | 450 | 50 | 50 | 2500 |

## 2.1. MAE (Mean Absolute Error, orta mütleq xeta)

**Menası.** En sade metrikdir: "model orta hesabla ne qeder yanılır?" Her evde xetanı götürüb işaresini atırıq (menfi ve müsbet xetalar bir-birini yox etmesin deye), sonra ortalamasını tapırıq.

**Düstur.** $MAE = \frac{1}{n}\sum |y_i - \hat{y}_i|$

**Nümunede hesablama.** (10 + 10 + 30 + 20 + 50) / 5 = 120 / 5 = **24**.

**Nece şerh olunur.** Ölçü vahidi qiymetin özü ile eynidir, yeni MAE = 24 min manat demekdir ki, model orta hesabla 24 min manat sehv edir. Bu, izah etmek üçün en rahat metrikdir. Kiçik olması yaxşıdır, 0 ideal haldır.

**Böyük xetalara reaksiyası.** Bütün xetalara beraber yanaşır. 50 min manatlıq sehv 10 min manatlıq sehvden düz 5 defe pis sayılır, ne az, ne çox. Ona göre kenar qiymetlere qarşı dözümlüdür.

## 2.2. MSE (Mean Squared Error, orta kvadratik xeta)

**Menası.** MAE-ye oxşardır, amma işareni atmaq üçün mütleq deyer evezine xetanı **kvadrata yükseldirik**, sonra ortalama alırıq.

**Düstur.** $MSE = \frac{1}{n}\sum (y_i - \hat{y}_i)^2$

**Nümunede hesablama.** (100 + 100 + 900 + 400 + 2500) / 5 = 4000 / 5 = **800**.

**Nece şerh olunur.** Vahid kvadrata yükselir: 800 "min manat kvadratı" demekdir, bunu birbaşa tesevvür etmek çetindir. Ona göre MSE-ni çox vaxt insanlara izah etmek üçün yox, modeli telim ve müqayise etmek üçün işledirler. Riyazi baxımdan rahatdır (hamar ve töremesi asan hesablanır), ona göre çox modelin teliminde hedef funksiyası kimi istifade olunur. Kiçik olması yaxşıdır.

**Böyük xetalara reaksiyası.** Burada böyük xetalar **sert cezalandırılır**. 50 min xeta kvadrata yükseldiyi üçün 2500 olur, 10 min xeta ise cemi 100. Yeni 5 defe böyük xeta 25 defe böyük ceza alır. Belelikle, bir-iki ağır sehv MSE-ni keskin artırır.

## 2.3. RMSE (Root Mean Squared Error, orta kvadratik xetanın kvadrat kökü)

**Menası.** MSE-nin kvadrat kökü. Meqsed MSE-nin üstünlüyünü (böyük xetaları görmek) saxlayıb, çatışmazlığını (vahidin kvadrat olması) düzeltmekdir.

**Düstur.** $RMSE = \sqrt{MSE}$

**Nümunede hesablama.** √800 ≈ **28.28**.

**Nece şerh olunur.** Vahid yene qiymetin özü ile eyni olur (min manat), ona göre MAE kimi başa düşülür: "model texminen 28 min manat yanılır". Amma bu "orta sehv" MAE-den (24) böyükdür, çünki böyük xetaları daha çox nezere alır. RMSE hemişe MAE-den böyük ve ya ona beraberdir. Eger RMSE MAE-den xeyli böyükdürse, bu, modelin arabir çox böyük sehvler etdiyine işaredir.

**Böyük xetalara reaksiyası.** MSE kimi, böyük xetalara hessasdır, amma kvadrat kök bu hessaslığı bir az yumşaldır.

## 2.4. R² (R-squared, determinasiya emsalı)

**Menası.** Evvelki üç metrik "neçe manat yanılırıq" sualına cavab verir. R² ise başqa sual verir: **model melumatdakı deyişkenliyin neçe faizini izah ede bilir?** Onu bele tesevvür edin: en sade "ağılsız" model heç neye baxmır ve hamıya orta qiymeti deyir. R² bizim modelin bu ağılsız modelden ne qeder yaxşı olduğunu gösterir.

**Düstur.** $R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$

Suret modelimizin ümumi kvadratik xetasıdır, mexrec ise "hamıya orta qiymeti deseydik" xetasıdır.

**Nümunede hesablama.** Heqiqi qiymetlerin ortası 300-dür. Orta ile ferqlerin kvadratı: 40000 + 10000 + 0 + 10000 + 40000 = 100000. Modelimizin xetalarının kvadratları cemi 4000-dir. Demeli R² = 1 − 4000/100000 = **0.96**.

**Nece şerh olunur.** R² = 1 mükemmel modeldir. R² = 0 demekdir ki, model sadece orta qiymeti deyen "ağılsız" model qeder yaxşıdır, yeni heç bir fayda yoxdur. R² menfi ola biler: bu, modelin orta qiymeti demekden de pis olduğunu gösterir. Bizim nümunede 0.96 onu gösterir ki, model qiymet deyişkenliyinin 96 faizini izah edir. R² vahidsizdir, ona göre müxtelif meseleleri müqayise etmek asandır. Amma tek başına kifayet etmir: R² yaxşı görünse de, manat ile xeta biznes üçün çox böyük ola biler. Ona göre onu MAE ve ya RMSE ile birlikde baxmaq lazımdır.

**Böyük xetalara reaksiyası.** Kvadrata esaslandığı üçün böyük xetalara hessasdır, MSE kimi. Böyük sehvler R²-ni süretle aşağı salır.

Gelin hesablamanı kodla yoxlayaq ve öz el hesablamamızın scikit-learn ile üst-üste düşdüyünü görek.

In [ ]:
hegiqi = np.array([100, 200, 300, 400, 500])
proqnoz = np.array([110, 190, 330, 380, 450])

xeta = hegiqi - proqnoz


mae_el = np.mean(np.abs(xeta))
mse_el = np.mean(xeta ** 2)
rmse_el = np.sqrt(mse_el)
r2_el = 1 - np.sum(xeta ** 2) / np.sum((hegiqi - hegiqi.mean()) ** 2)


mae_sk = mean_absolute_error(hegiqi, proqnoz)
mse_sk = mean_squared_error(hegiqi, proqnoz)
rmse_sk = np.sqrt(mse_sk)
r2_sk = r2_score(hegiqi, proqnoz)

pd.DataFrame({
    "El ile": [mae_el, mse_el, rmse_el, r2_el],
    "scikit-learn": [mae_sk, mse_sk, rmse_sk, r2_sk],
}, index=["MAE", "MSE", "RMSE", "R2"]).round(3)

### Böyük xetaya reaksiya: bir sehv her şeyi deyişir

İndi en maraqlı hisseye baxaq. Beşinci evde modelin proqnozunu 450-den 350-ye endirek, yeni xetanı 50-den 150-ye qaldıraq. Qalan dörd ev eyni qalır. Görek her metrik nece reaksiya verir.

In [ ]:
proqnoz_pis = proqnoz.copy()
proqnoz_pis[4] = 350

def hamisi(h, p):
    return {
        "MAE": mean_absolute_error(h, p),
        "MSE": mean_squared_error(h, p),
        "RMSE": np.sqrt(mean_squared_error(h, p)),
        "R2": r2_score(h, p),
    }

cedvel = pd.DataFrame({"Evvel (xeta 50)": hamisi(hegiqi, proqnoz),
                       "Sonra (xeta 150)": hamisi(hegiqi, proqnoz_pis)})
cedvel["Neçe defe artdı"] = cedvel["Sonra (xeta 150)"] / cedvel["Evvel (xeta 50)"]
cedvel.round(3)

Cedvelden aydın görünür ki, bir evde xeta 3 defe artanda MAE texminen 1.8 defe artır (yeni nisbeten mülayim reaksiya verir), RMSE texminen 2.4 defe, MSE ise 6 defe artır (kvadrata göre en sert reaksiya budur), R² ise 0.96-dan xeyli aşağı düşür. Bu, metriklerin xarakterini gösterir: MAE "bütün xetalar beraberdir" deyir, MSE ve RMSE ise "böyük sehvler çox pisdir" deyir.

Praktikada seçimi bele etmek olar. Eger her manatlıq sehv eyni derecede vacibdirse ve kenar qiymetler çoxdursa, MAE daha edaletlidir. Eger böyük sehvler xüsusile bahalıdırsa (meselen, evin qiymetini çox yüksek ve ya çox aşağı texmin etmek böyük itkiye sebeb olursa), RMSE daha uyğundur. Modelin ümumi keyfiyyetini vahidsiz görmek üçün R² baxılır. En yaxşı yol ise birini yox, birlikde baxmaqdır.

Dörd metrikin ümumi xülasesi: MAE orta xetadır, MSE böyük xetaları kvadratla cezalandırır, RMSE onu yeniden manat vahidine qaytarır, R² ise modelin "orta qiymet deyen ağılsız modelden ne qeder yaxşı olduğunu" gösterir.